## Final Project Submission

Please fill out:
* Student name: 
* Student pace: self paced / part time / full time
* Scheduled project review date/time: 
* Instructor name: 
* Blog post URL:


In [ ]:
# Your code here - remember to use markdown cells for comments as well!

## Business Problem

### The company `KENVENTURES` has decided to create a new movie studio `KENHOOD`, with no background about creating movies. 
### Find out what types of films are currently doing the best at the box office. Translate those findings into actionable insights that the board of director's new movie studio `KENWOOD` can use to help decide what type of films to create.

## Business Objectives
- Which genres deliver the best financial return? (measured by ROI and median profit) 

- Which genres do best with an international audience 

- Which studios perform best and do they align with the top genres

## Data Understanding

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

In [2]:
# Load the compressed CSV
bom_df = pd.read_csv("zippedData/bom.movie_gross.csv.gz")
bom_df.head()

,title,studio,domestic_gross,foreign_gross,year
0,Toy Story 3,BV,415000000.0,652000000,2010
1,Alice in Wonderland (2010),BV,334200000.0,691300000,2010
2,Harry Potter and the Deathly Hallows Part 1,WB,296000000.0,664300000,2010
3,Inception,WB,292600000.0,535700000,2010
4,Shrek Forever After,P/DW,238700000.0,513900000,2010


In [6]:
# Conect the SQLite database
conn = sqlite3.connect("zippedData/im.db")

# Inspect the tables available in the database
tables = pd.read_sql(
    '''
    SELECT name 
    FROM sqlite_master
    WHERE type= 'table' ;
''',
conn,)
print("Tables in im.db:")
display(tables)

Tables in im.db:


,name
0,movie_basics
1,directors
2,known_for
3,movie_akas
4,movie_ratings
5,persons
6,principals
7,writers


In [7]:
# Load `movie_basics` and `movie_ratings` tables into pandas DataFrame
movies_basics = pd.read_sql(
    '''
    SELECT * 
    FROM movie_basics
''',
conn,
)
movies_basics.head()

,movie_id,primary_title,original_title,start_year,runtime_minutes,genres
0,tt0063540,Sunghursh,Sunghursh,2013,175.0,"Action,Crime,Drama"
1,tt0066787,One Day Before the Rainy Season,Ashad Ka Ek Din,2019,114.0,"Biography,Drama"
2,tt0069049,The Other Side of the Wind,The Other Side of the Wind,2018,122.0,Drama
3,tt0069204,Sabse Bada Sukh,Sabse Bada Sukh,2018,NaN,"Comedy,Drama"
4,tt0100275,The Wandering Soap Opera,La Telenovela Errante,2017,80.0,"Comedy,Drama,Fantasy"


In [8]:
# Load `movie_basics` and `movie_ratings` tables into pandas DataFrame
movies_ratings = pd.read_sql(
    '''
    SELECT * 
    FROM movie_ratings
''',
conn,
)
movies_ratings.head()

,movie_id,averagerating,numvotes
0,tt10356526,8.3,31
1,tt10384606,8.9,559
2,tt1042974,6.4,20
3,tt1043726,4.2,50352
4,tt1060240,6.5,21


### Joining the three tables

### Data Inspection

##### 1. Structural and Dimensional assessment

In [11]:
# Understanding the format and column entries to preview the data
bom_df.head()


,title,studio,domestic_gross,foreign_gross,year
0,Toy Story 3,BV,415000000.0,652000000,2010
1,Alice in Wonderland (2010),BV,334200000.0,691300000,2010
2,Harry Potter and the Deathly Hallows Part 1,WB,296000000.0,664300000,2010
3,Inception,WB,292600000.0,535700000,2010
4,Shrek Forever After,P/DW,238700000.0,513900000,2010


In [12]:
bom_df.tail()

,title,studio,domestic_gross,foreign_gross,year
3382,The Quake,Magn.,6200.0,NaN,2018
3383,Edward II (2018 re-release),FM,4800.0,NaN,2018
3384,El Pacto,Sony,2500.0,NaN,2018
3385,The Swan,Synergetic,2400.0,NaN,2018
3386,An Actor Prepares,Grav.,1700.0,NaN,2018


In [13]:
# Checking the dimension of the data by determining the overall row and column counts
print(f"The DataFrame has {bom_df.shape[0]} Rows, and {bom_df.shape[1]} Columns")

The DataFrame has 3387 Rows, and 5 Columns


In [14]:
# Ispecting Metadata and Data Types by verifying column names, non-null counts and types
bom_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3387 entries, 0 to 3386
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   title           3387 non-null   object 
 1   studio          3382 non-null   object 
 2   domestic_gross  3359 non-null   float64
 3   foreign_gross   2037 non-null   object 
 4   year            3387 non-null   int64  
dtypes: float64(1), int64(1), object(3)
memory usage: 132.4+ KB


##### Data Types issues
- `foreign_gross` is stored as string instead of float
- `year` is stored as integer instead of datetime

##### 2. Statistical Profiling

In [15]:
# Numerical Summaeries 
bom_df.describe()

,domestic_gross,year
count,3.359000e+03,3387.000000
mean,2.874585e+07,2013.958075
std,6.698250e+07,2.478141
min,1.000000e+02,2010.000000
25%,1.200000e+05,2012.000000
50%,1.400000e+06,2014.000000
75%,2.790000e+07,2016.000000
max,9.367000e+08,2018.000000


In [ ]:
# # Categorical Summaries
# bom_df.describe(include=['0'])
# bom_df['category_column'].value_counts()

##### 3. Data Quality Verification

In [16]:
# Checking for missing values
missing = bom_df.isna().sum()

# Percentage of missing values
pct_missing = (missing/len(bom_df))*100

pd.DataFrame({'Missing Count': missing, 'Percentage (%)': pct_missing})


,Missing Count,Percentage (%)
title,0,0.000000
studio,5,0.147623
domestic_gross,28,0.826690
foreign_gross,1350,39.858282
year,0,0.000000


In [17]:
# Checking for Duplicate rows
print(f"Duplicate rows: {bom_df.duplicated().sum()}")

Duplicate rows: 0


In [ ]:
# # Check for anomalies and outliers
# sns.boxplot(x=bom_df['Categorical_colums'])

### Data Cleaning

##### 1. Standardizing Data Types

In [18]:
# Cleaning and convert <foreign_gross> to float
# Removing commas and white spaces and converting strig to interger
bom_df['foreign_gross'] = bom_df['foreign_gross'].astype(str).str.replace(',', '', regex=False)
bom_df['foreign_gross'] = pd.to_numeric(bom_df['foreign_gross'], errors='coerce')

# Converting `year` to datetime
# Converting the 4-digit integer into a datetime object
bom_df['year'] = pd.to_datetime(bom_df['year'].astype(str), format='%Y')

# Verifying the updated data types
bom_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3387 entries, 0 to 3386
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   title           3387 non-null   object        
 1   studio          3382 non-null   object        
 2   domestic_gross  3359 non-null   float64       
 3   foreign_gross   2037 non-null   float64       
 4   year            3387 non-null   datetime64[ns]
dtypes: datetime64[ns](1), float64(2), object(2)
memory usage: 132.4+ KB
